# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [19]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [20]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [21]:
#your code here
initial_data=spaceship.shape[0]
initial_data

8693

**Check for data types**

In [22]:
#your code here
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


**Check for missing values**

In [23]:
#your code here
spaceship["CryoSleep"].isnull().value_counts()

CryoSleep
False    8476
True      217
Name: count, dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [24]:
#Se eliminan todas las filas con al menos un valor nulo en las columnas
spaceship_1 = spaceship.dropna()
datos_totales=initial_data-spaceship_1.shape[0]
datos_totales
print(f"Se mantuvo un total de {datos_totales} datos, es decir, el {round(datos_totales/initial_data*100,2)} % del total inicial")

Se mantuvo un total de 2087 datos, es decir, el 24.01 % del total inicial


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [25]:
# Extraer la primera letra de la columna "Cabin"
spaceship_1.loc[:, 'Cabin'] = spaceship_1['Cabin'].str.strip().str[0]

- Drop PassengerId and Name

In [26]:
#your code here
spaceship_1= spaceship_1.drop(columns=["PassengerId", "Name"])

- For non-numerical columns, do dummies.

In [27]:
# One-hot encoding for sex and title
spaceship_1= pd.get_dummies(spaceship_1, columns=['HomePlanet','Cabin','Destination']) #las binarias no se incluyen

In [28]:
spaceship_1.sample(4)

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,...,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
3410,False,34.0,False,232.0,457.0,1.0,218.0,0.0,True,True,...,False,False,False,False,False,True,False,True,False,False
3186,False,26.0,False,6.0,0.0,400.0,2.0,373.0,False,True,...,False,False,False,False,False,True,False,False,True,False
3598,False,9.0,False,0.0,0.0,0.0,0.0,0.0,True,True,...,False,False,False,False,False,True,False,False,False,True
6111,True,42.0,False,0.0,0.0,0.0,0.0,0.0,True,False,...,False,True,False,False,False,False,False,True,False,False


**Perform Train Test Split**

#### Quick EDA

In [29]:
import plotly.express as px
import plotly.graph_objects as go

# Se genera la matriz de correlación usando Spearman
correlation_matrix = spaceship_1.corr(method='spearman')

# Crea el gráfico de correlación utilizando plotly.graph_objects
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    colorbar=dict(title='Correlación', tickvals=[-1, 0, 1], ticktext=['-1', '0', '1']),
))

# Ajustar título y mostrar el gráfico
fig.update_layout(
    title='Matriz de Correlación Spearman',
    xaxis_title='Variables',
    yaxis_title='Variables',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    width=800, height=700
)

fig.show()

In [30]:
#Se generan dataframes para características y targets
features = spaceship_1.drop(columns = ["Transported"])
target = spaceship_1["Transported"]

#Se ejecuta la división de datos para entrenar y probar el modelo, considerando X para features (variables independientes)
#  y Y para target (variable dependiente a predecir). Se asigna el 80% de los datos para entrenar el modelo, 
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=42)

Para poder analizar con las variables que se hayan correlacionadas entre sí, se aplica análisis de componentes principales PCA, para unificar en un único vector de componentes principales, conservando así la mayor parte de la información (varianza) posible.
Se realizarán 5 PCA para las variables

In [31]:
#PCA para reducir la dimnesión de destination 'Destination_55 Cancri e', 'Destination_PSO J318.5-22','Destination_TRAPPIST-1e'
from sklearn.decomposition import PCA
pca_train_Destination=PCA(n_components=1)
pca_train_Destination.fit(X_train[['Destination_55 Cancri e', 'Destination_PSO J318.5-22','Destination_TRAPPIST-1e']])
X_train['Destination_PCA']=pca_train_Destination.transform(X_train[['Destination_55 Cancri e', 'Destination_PSO J318.5-22','Destination_TRAPPIST-1e']])
X_test['Destination_PCA']=pca_train_Destination.transform(X_test[['Destination_55 Cancri e', 'Destination_PSO J318.5-22','Destination_TRAPPIST-1e']])
#spaceship_2=spaceship_2.drop(columns=['Destination_55 Cancri e', 'Destination_PSO J318.5-22','Destination_TRAPPIST-1e'])


In [32]:
#PCA para reducir la dimnesión de 'HomePlanet_Earth', 'HomePlanet_Europa','HomePlanet_Mars'
columns=['Cabin_A','Cabin_B', 'Cabin_C', 'Cabin_D', 'Cabin_E', 'Cabin_F', 'Cabin_G','Cabin_T']
from sklearn.decomposition import PCA
pca_train_planet=PCA(n_components=0.7)
pca_train_planet.fit(X_train[columns])
pca_result_train=pca_train_planet.transform(X_train[columns])

num_components = pca_result_train.shape[1]

# Crear nombres de columnas genéricos basados en el número de componentes
pca_column_names = [f'PCA_cabin_{i+1}' for i in range(num_components)]

# Agregar las nuevas columnas al DataFrame X_train
X_train[pca_column_names] = pca_result_train

pca_result_test=pca_train_planet.transform(X_test[columns])

num_components = pca_result_test.shape[1]

# Crear nombres de columnas genéricos basados en el número de componentes
pca_column_names = [f'PCA_cabin_{i+1}' for i in range(num_components)]

# Agregar las nuevas columnas al DataFrame X_train
X_test[pca_column_names] = pca_result_test

In [33]:
X_test.sample(4)

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,...,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_PCA,PCA_cabin_1,PCA_cabin_2,PCA_cabin_3
3642,False,14.0,False,2.0,881.0,0.0,578.0,214.0,False,True,...,False,False,False,False,False,True,0.384684,-0.038669,-0.385955,-0.076050
2313,True,27.0,False,0.0,0.0,0.0,0.0,0.0,False,False,...,True,False,False,False,False,True,0.384684,0.700509,0.284502,0.014003
4608,False,18.0,False,0.0,0.0,0.0,702.0,0.0,True,False,...,True,False,False,False,False,True,0.384684,0.700509,0.284502,0.014003
8146,False,28.0,False,679.0,0.0,0.0,0.0,0.0,True,False,...,False,True,False,False,True,False,-0.529145,-0.712317,0.347058,0.015964


In [34]:
#PCA para reducir la dimnesión de 'HomePlanet_Earth', 'HomePlanet_Europa','HomePlanet_Mars'
columns=['HomePlanet_Earth', 'HomePlanet_Europa','HomePlanet_Mars']
from sklearn.decomposition import PCA
pca_train_planet=PCA(n_components=1)
pca_train_planet.fit(X_train[columns])
X_train['homeplanet_PCA']=pca_train_planet.transform(X_train[columns])
X_test['homeplanet_PCA']=pca_train_planet.transform(X_test[columns])

In [35]:
X_train.columns

Index(['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall',
       'Spa', 'VRDeck', 'HomePlanet_Earth', 'HomePlanet_Europa',
       'HomePlanet_Mars', 'Cabin_A', 'Cabin_B', 'Cabin_C', 'Cabin_D',
       'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_T', 'Destination_55 Cancri e',
       'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e',
       'Destination_PCA', 'PCA_cabin_1', 'PCA_cabin_2', 'PCA_cabin_3',
       'homeplanet_PCA'],
      dtype='object')

In [36]:
features_selected=['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall',
       'Spa', 'VRDeck','Destination_PCA', 'PCA_cabin_1', 'PCA_cabin_2', 'PCA_cabin_3',
       'homeplanet_PCA']

In [37]:
# scaling the data 
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler= StandardScaler()

scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [41]:
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

knn = KNeighborsClassifier(n_neighbors=10)

knn.fit(X_train, y_train)

pred = knn.predict(X_test)
pred

# Assuming 'y_test' are the true labels and 'pred' are the model's predictions


# Complete classification report
report = classification_report(y_test, pred)
print(report)

              precision    recall  f1-score   support

       False       0.79      0.83      0.81       653
        True       0.82      0.79      0.80       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



In [ ]:
"""
Tengo variables:
Numéricas continuas
Binarias (provenientes de object y resultantes de get_dummies)
El análisis de correlación lineal aplicado a todo, es aceptable?

Divido datos en features y target
Obtengo df de X_train, X_test, etc.

--> aquí entra la estandarización y normalización

SI aplico PCA para generar un único vector para las variables VIP:
-VIP, proviene de 2 variables binarias. ¿Se estandariza y normaliza? NO, i solo incluyo una de ellas

nunca normalizar ni normalizar antes de Split

-Cabin, proviene de 8 variables binarias. ¿Se estandariza y normaliza? no se tienen que normalizar ni estandarizan, tampoco los datos que se tratan de lo mismo. (precios entre precios, etc)

El PCA se entrena con X_train y se aplica a Xtrain y Xtest?

cuando tengo dos cstegorias binarias, elimino una de ellas cuando están tan relacionadas

EL PCA SE APLICA DESPUÉS DEL SPLIT, EL PCA SE GUARDA EN PKL

"""

- Evaluate your model's performance. Comment it

In [16]:
#your code here